## Path

In [ ]:
import sys
import os

backend_path = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, backend_path)

print("Added to path:", backend_path)
print("Exists:", os.path.isdir(backend_path))

Added to path: c:\Users\DELL\Desktop\medrag\backend
Exists: True


## The Article Data Model

In [2]:
from pydantic import BaseModel
from typing import List


class Article(BaseModel):
    pmid: str
    title: str
    abstract: str
    authors: List[str]
    journal: str
    pub_date: str
    language: str
    topic: str
    source: str = "pubmed"
    url: str

## Test the Model Works

In [3]:
test_article = Article(
    pmid="12345678",
    title="A Study on Diabetes Management",
    abstract="This is a sample abstract about diabetes.",
    authors=["Smith J", "Doe A"],
    journal="Journal of Medicine",
    pub_date="2023-05-01",
    language="eng",
    topic="diabetes",
    url="https://pubmed.ncbi.nlm.nih.gov/12345678/"
)

print(test_article)

pmid='12345678' title='A Study on Diabetes Management' abstract='This is a sample abstract about diabetes.' authors=['Smith J', 'Doe A'] journal='Journal of Medicine' pub_date='2023-05-01' language='eng' topic='diabetes' source='pubmed' url='https://pubmed.ncbi.nlm.nih.gov/12345678/'


## Connect to PubMed via Biopython

In [ ]:
from Bio import Entrez                             # Instead of writing complicated HTTP requests yourself, Entrez does it for you
from config.settings import settings

Entrez.email = settings.pubmed_email               # Whenever you contact PubMed, use this email
if settings.ncbi_api_key:
    Entrez.api_key = settings.ncbi_api_key

print("Entrez configured with email:", Entrez.email)
print("API key set:", bool(settings.ncbi_api_key))

Entrez configured with email: abdulmoiz28.7.2002@gmail.com
API key set: True


## ESearch — Search PubMed for Article IDs

In [5]:
def search_pubmed(topic: str, max_results: int = 130):
    """
    Search PubMed for a topic, return a list of PMIDs.
    """
    handle = Entrez.esearch(db="pubmed", term=topic, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]


# Test with just one topic, small number first
test_pmids = search_pubmed("diabetes", max_results=5)
print("Found PMIDs:", test_pmids)
print("Count:", len(test_pmids))

Found PMIDs: ['42480168', '42480128', '42480103', '42480062', '42480009']
Count: 5


## EFetch — Fetch Full Article Data for Those PMIDs

In [ ]:
def fetch_articles_raw(pmids: list):
    """
    Fetch full article records for a list of PMIDs.
    Returns Biopython's parsed structure (nested dicts/lists).
    """
    ids = ",".join(pmids)
    handle = Entrez.efetch(db="pubmed", id=ids, rettype="abstract", retmode="xml")
    records = Entrez.read(handle)
    handle.close()
    return records


# Test with the 5 PMIDs from Cell 4
raw_records = fetch_articles_raw(test_pmids)

# Just inspect the very first record's top-level keys for now — don't try to parse yet
print(type(raw_records))
print(raw_records.keys())            # This means the record has two main sections 
print(len(raw_records["PubmedArticle"]))

<class 'Bio.Entrez.Parser.DictionaryElement'>
dict_keys(['PubmedBookArticle', 'PubmedArticle'])
5


## Inspect One Raw Record's Structure

In [ ]:
first_article = raw_records["PubmedArticle"][0]

# Print the top-level structure so we can see where things live
print(first_article.keys())                   # This means the record has two main sections
print("---")
print(first_article["MedlineCitation"].keys()) # Most of the information you need (title, abstract etc.) is inside MedlineCitation
# This article variable contains the information we're interested in

dict_keys(['MedlineCitation', 'PubmedData'])
---
dict_keys(['GeneralNote', 'InvestigatorList', 'CitationSubset', 'OtherID', 'OtherAbstract', 'SpaceFlightMission', 'KeywordList', 'PMID', 'DateRevised', 'Article', 'MedlineJournalInfo', 'CoiStatement'])


## Inspect the Article Sub-Structure

In [8]:
article_data = first_article["MedlineCitation"]["Article"]
print(article_data.keys())

dict_keys(['ELocationID', 'ArticleDate', 'Language', 'Journal', 'ArticleTitle', 'Pagination', 'Abstract', 'AuthorList', 'PublicationTypeList'])


## Extract the Title and Language

In [9]:
title = str(article_data["ArticleTitle"])
language = article_data["Language"][0]  # it's a list, usually with one entry like 'eng'

print("Title:", title)
print("Language:", language)

Title: Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.
Language: eng


## Extract the Abstract

In [10]:
abstract_parts = article_data["Abstract"]["AbstractText"]

# abstract_parts is a list — could be one long string, or multiple labeled sections
abstract = " ".join(str(part) for part in abstract_parts)

print("Abstract:")
print(abstract)
print("---")
print("Number of sections:", len(abstract_parts))

Abstract:
First-degree relatives (FDRs) of type 1 diabetes (T1D) patients are at high risk for T1D. Early screening enables timely intervention, yet participation remains suboptimal, missing opportunities in this high-risk group. Understanding determinants of screening participation and translating them into actionable, theory-informed, family-centered strategies is essential to optimize preventive care. In this qualitative study in China, semi-structured interviews were conducted with 21 adult T1D probands or parents of pediatric T1D patients who have unaffected children. The Theoretical Domains Framework (TDF) guided data collection and analysis. Barriers were mapped to Behavior Change Techniques (BCTs). We identified 18 facilitators, 11 barriers, and 2 neutral factors, mapped across 12 TDF domains. Key barriers included limited awareness, misconceptions, cost, restricted medical resources, insufficient publicity, and anxiety; key facilitators included perceived benefits, optimism, h

## Extract Authors

In [11]:
def extract_authors(article_data):
    authors = []
    author_list = article_data.get("AuthorList", [])
    for author in author_list:
        if "LastName" in author and "ForeName" in author:
            authors.append(f"{author['LastName']} {author['ForeName']}")
        elif "CollectiveName" in author:
            authors.append(str(author["CollectiveName"]))
    return authors


authors = extract_authors(article_data)
print("Authors:", authors)
print("Count:", len(authors))

Authors: ['Li Zhisheng', 'Zeng Hanyue', 'Zhou Mengna', 'Deng Chao', 'Guo Jia', 'Li Xia', 'Xie Yuting', 'Yan Junxia']
Count: 8


## Extract Journal Name and Publication Date

In [12]:
def extract_journal_and_date(article_data):
    journal = str(article_data["Journal"]["Title"])
    
    pub_date_data = article_data["Journal"]["JournalIssue"]["PubDate"]
    
    year = pub_date_data.get("Year", "")
    month = pub_date_data.get("Month", "")
    day = pub_date_data.get("Day", "")
    
    # Build date string from whatever parts are available
    date_parts = [p for p in [year, month, day] if p]
    pub_date = "-".join(date_parts) if date_parts else "unknown"
    
    return journal, pub_date


journal, pub_date = extract_journal_and_date(article_data)
print("Journal:", journal)
print("Pub Date:", pub_date)

Journal: Journal of pediatric nursing
Pub Date: 2026-Jul-21


## Assemble Everything Into One parse_article() Function

In [13]:
def parse_article(pubmed_article: dict, topic: str) -> Article:
    medline = pubmed_article["MedlineCitation"]
    article_data = medline["Article"]
    pmid = str(medline["PMID"])
    
    title = str(article_data["ArticleTitle"])
    language = article_data["Language"][0]
    
    abstract_parts = article_data.get("Abstract", {}).get("AbstractText", [])
    abstract = " ".join(str(part) for part in abstract_parts) if abstract_parts else ""
    
    authors = extract_authors(article_data)
    journal, pub_date = extract_journal_and_date(article_data)
    
    return Article(
        pmid=pmid,
        title=title,
        abstract=abstract,
        authors=authors,
        journal=journal,
        pub_date=pub_date,
        language=language,
        topic=topic,
        url=f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    )


# Test on the same first_article we've been inspecting
parsed = parse_article(first_article, topic="diabetes")
print(parsed)

pmid='42480168' title='Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.' abstract='First-degree relatives (FDRs) of type 1 diabetes (T1D) patients are at high risk for T1D. Early screening enables timely intervention, yet participation remains suboptimal, missing opportunities in this high-risk group. Understanding determinants of screening participation and translating them into actionable, theory-informed, family-centered strategies is essential to optimize preventive care. In this qualitative study in China, semi-structured interviews were conducted with 21 adult T1D probands or parents of pediatric T1D patients who have unaffected children. The Theoretical Domains Framework (TDF) guided data collection and analysis. Barriers were mapped to Behavior Change Techniques (BCTs). We identified 18 facilitators, 11 barriers, and 2 neutral factors, mapped across 12 TDF domains. Key barriers included limited awa

## Parse All 5 Test Articles + Handle Errors Safely

In [14]:
def parse_articles_safe(raw_articles: list, topic: str):
    parsed = []
    failed = []
    
    for raw in raw_articles:
        try:
            article = parse_article(raw, topic=topic)
            parsed.append(article)
        except Exception as e:
            pmid = raw.get("MedlineCitation", {}).get("PMID", "unknown")
            failed.append({"pmid": str(pmid), "error": str(e)})
    
    return parsed, failed


parsed_articles, failed_articles = parse_articles_safe(raw_records["PubmedArticle"], topic="diabetes")

print(f"Parsed: {len(parsed_articles)}")
print(f"Failed: {len(failed_articles)}")
if failed_articles:
    print("Failures:", failed_articles)

Parsed: 5
Failed: 0


## Save Articles to Disk (JSONL format)

In [15]:
import json
from pathlib import Path

def save_articles(articles: list, topic: str, output_dir: str = "../data/raw/pubmed"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = Path(output_dir) / f"{topic}.jsonl"
    
    with open(filepath, "a", encoding="utf-8") as f:
        for article in articles:
            f.write(article.model_dump_json() + "\n")
    
    return filepath


saved_path = save_articles(parsed_articles, topic="diabetes")
print("Saved to:", saved_path)
print("Absolute path:", saved_path.resolve())

Saved to: ..\data\raw\pubmed\diabetes.jsonl
Absolute path: C:\Users\DELL\Desktop\medrag\data\raw\pubmed\diabetes.jsonl


## Verify by Reading the File Back

In [16]:
def load_articles(topic: str, output_dir: str = "../data/raw/pubmed"):
    filepath = Path(output_dir) / f"{topic}.jsonl"
    articles = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            articles.append(Article(**data))
    return articles


loaded_articles = load_articles("diabetes")
print(f"Loaded {len(loaded_articles)} articles")
print(loaded_articles[0].title)

Loaded 5 articles
Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.


## Idempotency Check — Skip Already-Downloaded PMIDs

In [17]:
def get_existing_pmids(topic: str, output_dir: str = "../data/raw/pubmed"):
    filepath = Path(output_dir) / f"{topic}.jsonl"
    if not filepath.exists():
        return set()
    
    existing = set()
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            existing.add(data["pmid"])
    return existing


existing = get_existing_pmids("diabetes")
print("Existing PMIDs for diabetes:", existing)
print("Count:", len(existing))

Existing PMIDs for diabetes: {'42480062', '42480128', '42480009', '42480168', '42480103'}
Count: 5


## Add Rate Limiting

In [18]:
import time

def rate_limit_delay(has_api_key: bool = True):
    """
    Pause briefly between requests to respect PubMed's rate limits.
    ~10 req/sec with API key (0.1s delay), ~3 req/sec without (0.34s delay).
    """
    delay = 0.11 if has_api_key else 0.35
    time.sleep(delay)


# Test it doesn't error
rate_limit_delay(has_api_key=bool(settings.ncbi_api_key))
print("Delay executed without error")

Delay executed without error


## The Full Ingestion Pipeline

In [19]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("medrag.ingestion")


def ingest_topic(topic: str, target_count: int = 130):
    """
    Full pipeline for one topic: search, skip existing, fetch, parse, save.
    """
    logger.info(f"Starting ingestion for topic: {topic}")

    # 1. Check what we already have
    existing_pmids = get_existing_pmids(topic)
    logger.info(f"  Already have {len(existing_pmids)} articles for '{topic}'")

    # 2. Search for PMIDs (search for more than target in case some overlap with existing)
    search_count = target_count + len(existing_pmids)
    all_pmids = search_pubmed(topic, max_results=search_count)
    rate_limit_delay(has_api_key=bool(settings.ncbi_api_key))

    # 3. Filter out ones we already have
    new_pmids = [p for p in all_pmids if p not in existing_pmids][:target_count]
    logger.info(f"  {len(new_pmids)} new articles to fetch")

    if not new_pmids:
        logger.info(f"  Nothing new to fetch for '{topic}'")
        return {"topic": topic, "fetched": 0, "failed": 0, "skipped": len(existing_pmids)}

    # 4. Fetch in one batch
    raw_records = fetch_articles_raw(new_pmids)
    rate_limit_delay(has_api_key=bool(settings.ncbi_api_key))

    # 5. Parse safely
    parsed, failed = parse_articles_safe(raw_records["PubmedArticle"], topic=topic)

    # 6. Save
    if parsed:
        save_articles(parsed, topic=topic)

    logger.info(f"  Saved {len(parsed)}, failed {len(failed)} for '{topic}'")
    if failed:
        logger.warning(f"  Failures: {failed}")

    return {"topic": topic, "fetched": len(parsed), "failed": len(failed), "skipped": len(existing_pmids)}


# Test on a fresh topic we haven't touched yet
result = ingest_topic("hypertension", target_count=10)
print(result)

2026-07-22 17:19:59,769 - INFO - Starting ingestion for topic: hypertension
2026-07-22 17:19:59,771 - INFO -   Already have 0 articles for 'hypertension'
2026-07-22 17:20:01,243 - INFO -   10 new articles to fetch
2026-07-22 17:20:02,627 - INFO -   Saved 10, failed 0 for 'hypertension'


{'topic': 'hypertension', 'fetched': 10, 'failed': 0, 'skipped': 0}


## Run the Full Batch — All 32 Topics at Target Count

In [20]:
topics = [
    "diabetes", "hypertension", "obesity", "hyperlipidemia",
    "asthma", "copd", "pneumonia", "tuberculosis",
    "coronary artery disease", "heart failure", "stroke", "arrhythmia",
    "malaria", "dengue fever", "hiv aids", "hepatitis b", "covid-19", "typhoid",
    "depression", "anxiety disorder",
    "peptic ulcer disease", "irritable bowel syndrome", "hepatitis c",
    "osteoarthritis", "rheumatoid arthritis", "osteoporosis",
    "hypothyroidism", "hyperthyroidism",
    "epilepsy", "migraine", "parkinson's disease",
    "chronic kidney disease",
    "breast cancer", "lung cancer",
    "anemia in pregnancy", "malnutrition"
]

print(f"Total topics: {len(topics)}")

all_results = []

for topic in topics:
    result = ingest_topic(topic, target_count=130)
    all_results.append(result)

# Summary
total_fetched = sum(r["fetched"] for r in all_results)
total_failed = sum(r["failed"] for r in all_results)
print(f"\n=== DONE ===")
print(f"Total articles fetched: {total_fetched}")
print(f"Total failed: {total_failed}")

2026-07-22 17:20:39,200 - INFO - Starting ingestion for topic: diabetes
2026-07-22 17:20:39,203 - INFO -   Already have 5 articles for 'diabetes'


Total topics: 36


2026-07-22 17:20:40,501 - INFO -   130 new articles to fetch
2026-07-22 17:20:42,948 - INFO -   Saved 130, failed 0 for 'diabetes'
2026-07-22 17:20:42,957 - INFO - Starting ingestion for topic: hypertension
2026-07-22 17:20:42,958 - INFO -   Already have 10 articles for 'hypertension'
2026-07-22 17:20:44,123 - INFO -   130 new articles to fetch
2026-07-22 17:20:46,551 - INFO -   Saved 130, failed 0 for 'hypertension'
2026-07-22 17:20:46,551 - INFO - Starting ingestion for topic: obesity
2026-07-22 17:20:46,552 - INFO -   Already have 0 articles for 'obesity'
2026-07-22 17:20:47,666 - INFO -   130 new articles to fetch
2026-07-22 17:20:49,922 - INFO -   Saved 130, failed 0 for 'obesity'
2026-07-22 17:20:49,923 - INFO - Starting ingestion for topic: hyperlipidemia
2026-07-22 17:20:49,924 - INFO -   Already have 0 articles for 'hyperlipidemia'
2026-07-22 17:20:50,813 - INFO -   130 new articles to fetch
2026-07-22 17:20:54,558 - INFO -   Saved 128, failed 0 for 'hyperlipidemia'
2026-07-22


=== DONE ===
Total articles fetched: 4661
Total failed: 0
